# CPM-IAT Schaefer200 + Ridge nested CV

使用 Schaefer200 cortical ROI 与 Harvard-Oxford 14 个 subcortical ROI 的功能连接预测 `IAT_PCIAT_Total`。

- 分组：`highIAT` 和 `lowIAT`
- 外层验证：LOOCV
- 内层验证：KFold，选择 `ridge_alpha`
- 选边：训练集内偏相关检验
- 模型：以入选 edge 为特征的 Ridge 回归
- 协变量：年龄、性别、父母教育与职业、CBCL 内外化、头动和扫描站点


In [1]:
# ============================================================
# 0. 线程控制：需要在 numpy/scipy/sklearn 大量调用前设置
# ============================================================
# 说明：biox.CPM 当前是简洁的串行 nested CV 实现。限制 BLAS 线程可以避免
# 在 notebook 或多任务环境中出现过度抢占 CPU 的情况。

import os

MAX_BLAS_THREADS = 1

os.environ["OMP_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["MKL_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(MAX_BLAS_THREADS)

print("MAX_BLAS_THREADS=", MAX_BLAS_THREADS)

MAX_BLAS_THREADS= 1


In [2]:
# ============================================================
# 1. Imports：优先调用 DATA_ROOT/biox 中已经封装好的 CPM 工具
# ============================================================
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 数据和脚本都放在 DATA_ROOT 下；后续路径都从这里派生。
DATA_ROOT = Path("/Users/fmri24/Desktop/YiZhang")

# biox package 位于 DATA_ROOT/biox；导入 package 时需要把父目录 DATA_ROOT 加入 sys.path。
if str(DATA_ROOT) not in sys.path:
    sys.path.insert(0, str(DATA_ROOT))

from biox.CPM import cpm_nested_cv_parallel, calculate_prediction_performance
from biox.fcmat import edgeList
from biox.prep import fcMatrixGp

print("DATA_ROOT:", DATA_ROOT)
print("Imported biox.CPM.cpm_nested_cv_parallel, biox.fcmat.edgeList, and biox.prep.fcMatrixGp")

DATA_ROOT: /Users/fmri24/Desktop/YiZhang
Imported biox.CPM.cpm_nested_cv_parallel, biox.fcmat.edgeList, and biox.prep.fcMatrixGp


## 2. 路径与参数

`DATA_ROOT` 是量表、功能时间序列和 `biox` package 的共同根目录。结果保存在 `RESULT_DIR`。

In [3]:
# ============================================================
# 2. 路径与全局参数
# ============================================================
SCALE_CSV = DATA_ROOT / "量表数据汇总_with_MeanEnorm.csv"

# 功能数据目录：Schaefer200 cortical ROI + Harvard-Oxford subcortical ROI。
SCHAEFER_DIR = DATA_ROOT / "HBN_func_Schaefer200"
HO_DIR = DATA_ROOT / "HBN_func_HarvardOxfordSubcortical"

# Harvard-Oxford 原始下标，原 notebook 使用 1-based 编号后减 1。
HO_RAW_INDEX = np.array([4, 5, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21]) - 1

# 量表和协变量列名。
IAT_LABEL = "IAT_PCIAT_Total"
SITE_COL = "ScannerSite"
CBCL_TOTAL_COL = "CBCL,CBCL_Total_T"
MOTION_COL = "mean_enorm"

COVARIATE_COLS = [
    "Basic_Demos,Age",
    "Basic_Demos,Sex",
    "Barratt,Barratt_Total_Edu",
    "Barratt,Barratt_Total_Occ",
    "CBCL,CBCL_Int_T",
    "CBCL,CBCL_Ext_T",
    MOTION_COL,
]

EXPECTED_N_TIMEPOINTS = 740
IAT_CUTOFF = 30
CBCL_CUTOFF = 70
ENORM_CUTOFF = 0.3
SUBGROUPS = ["highIAT", "lowIAT"]

# nested CV 结果目录。
RESULT_DIR = DATA_ROOT / "CPM_nestedCV_LOOCV"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# CPM / nested CV 主分析参数。
P_THRESH = 0.01
INNER_N_SPLITS = 5
RANDOM_STATE = 2026
N_JOBS = 20
STANDARDIZE_FEATURES = True

# Ridge alpha 网格：log-scale 搜索，并强制包含一组重点关注的 alpha。
# ALPHA_GRID = np.unique(np.concatenate([
#     np.logspace(-3, 4, 57),
#     np.array([0.03, 0.05623413, 0.1, 0.31622777, 1.0, 3.16227766, 10.0, 31.6227766, 100.0])
# ])).astype(float)

ALPHA_GRID = np.logspace(-2, 2, 100)
ALPHA_GRID = np.sort(ALPHA_GRID)

print("DATA_ROOT:", DATA_ROOT)
print("SCALE_CSV:", SCALE_CSV)
print("RESULT_DIR:", RESULT_DIR)
print("n_alpha:", len(ALPHA_GRID), "alpha range:", (ALPHA_GRID.min(), ALPHA_GRID.max()))

DATA_ROOT: /Users/fmri24/Desktop/YiZhang
SCALE_CSV: /Users/fmri24/Desktop/YiZhang/量表数据汇总_with_MeanEnorm.csv
RESULT_DIR: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV
n_alpha: 100 alpha range: (np.float64(0.01), np.float64(100.0))


## 3. FC 计算

每名被试读取 `func1` 和 `func2`，拼接 Schaefer200 与 HO14 时间序列。`fcMatrixGp()` 计算 Fisher-z FC 矩阵，`edgeList()` 提取上三角 edge，最终得到与 `df_atlas` 按行对应的 `FC_all`。

In [4]:
# ============================================================
# 3. FC 计算 / 数据读取
# ============================================================
# 本单元直接从原始时间序列计算 CPM 需要的 FC edge matrix。
# 这里不再定义函数：按顺序执行，比包一层单次使用的函数更容易阅读。

# 1) 读取量表总表。后续所有筛选和对齐都以这个表为起点。
df_atlas = pd.read_csv(SCALE_CSV)

# 2) 统一 ID 类型，并根据 ID 生成两段功能时间序列的文件名。
#    原始数据约定每个被试有两个 run：func1 和 func2。
df_atlas["ID"] = df_atlas["ID"].astype(str)
df_atlas["func1"] = "sub-" + df_atlas["ID"] + "_func1.npy"
df_atlas["func2"] = "sub-" + df_atlas["ID"] + "_func2.npy"

# 3) 只保留两个 atlas 目录中都存在的文件。
#    files_present_in_both_atlases 表示“两个 atlas 文件夹都能找到的 .npy 文件名”。
#    Schaefer200 和 HO14 都要有 func1/func2，后面才能把 cortical 和 subcortical ROI 拼起来。
files_present_in_both_atlases = set(os.listdir(SCHAEFER_DIR)) & set(os.listdir(HO_DIR))
df_atlas = df_atlas[
    df_atlas["func1"].isin(files_present_in_both_atlases)
    & df_atlas["func2"].isin(files_present_in_both_atlases)
].reset_index(drop=True)

# 4) 逐被试读取时间序列、检查时间点数，并计算 Fisher-z FC edge vector。
#    这里把“时间点检查”和“FC 计算”放在同一轮循环里，避免同一个 .npy 文件被读取两次。
#    每个被试的处理顺序如下：
#    a. 读取 Schaefer200 和 HO 的 func1/func2；
#    b. 检查两个 run 拼接后的总时间点是否等于 EXPECTED_N_TIMEPOINTS；
#    c. 通过检查后，沿时间维 axis=1 拼接两个 run；
#    d. HO 只保留 HO_RAW_INDEX 指定的 14 个 ROI；
#    e. 沿 ROI 维 axis=0 拼接 Schaefer200 和 HO14，得到 214 ROI x 740 time points；
#    f. 用 biox.prep.fcMatrixGp 计算相关矩阵并做 Fisher z；
#    g. 用 biox.fcmat.edgeList 取上三角 edge，得到一行 CPM 特征。
fc_list = []
timepoint_ok_row_idx = []
for i in range(df_atlas.shape[0]):
    f1 = df_atlas.loc[i, "func1"]
    f2 = df_atlas.loc[i, "func2"]

    sch1 = np.load(SCHAEFER_DIR / f1)
    sch2 = np.load(SCHAEFER_DIR / f2)
    ho1 = np.load(HO_DIR / f1)
    ho2 = np.load(HO_DIR / f2)

    if (sch1.shape[1] + sch2.shape[1] != EXPECTED_N_TIMEPOINTS) or (ho1.shape[1] + ho2.shape[1] != EXPECTED_N_TIMEPOINTS):
        continue

    ts_sch = np.concatenate([sch1, sch2], axis=1)
    ts_ho = np.concatenate([ho1, ho2], axis=1)[HO_RAW_INDEX, :]
    ts = np.concatenate([ts_sch, ts_ho], axis=0)

    fc_z = fcMatrixGp(ts[np.newaxis, :, :], fisherZ=True)[0]
    fc_list.append(edgeList(fc_z, k=1))
    timepoint_ok_row_idx.append(i)

# 5) 只保留真正进入 FC 计算的被试，并堆叠成 CPM 输入矩阵。
#    timepoint_ok_row_idx 记录“时间点数通过检查、并已成功计算 FC”的 df_atlas 行号。
#    df_atlas 的行顺序和 FC_all 的行顺序一一对应。
df_atlas = df_atlas.iloc[timepoint_ok_row_idx].reset_index(drop=True)
FC_all = np.asarray(fc_list, dtype=np.float32)
print("FC_all:", FC_all.shape)


/Users/fmri24/Desktop/YiZhang/biox/prep.py:68: RuntimeWarning: divide by zero encountered in arctanh
  fcMatrixGp = np.arctanh(fcMatrixGp)


FC_all: (1376, 22791)


## 4. 运行 nested LOOCV 主分析

`cpm_nested_cv_parallel()` 将彼此独立的 outer folds 分配给多个进程。每个 outer training set 内完成协变量残差化、选边和 inner-CV 调参，再预测未参与训练与调参的 outer test subject：

```python
cv_results = cpm_nested_cv_parallel(..., n_jobs=N_JOBS)
performance = calculate_prediction_performance(cv_results[3], cv_results[2])
```

In [5]:
# ============================================================
# 4. 运行 highIAT / lowIAT 的 biox nested CPM
# ============================================================
# 这一单元只负责准备数据、调用 biox.CPM，并保留函数的原始输出。
# 不把 cv_results 中的 selected_param_records、edge_records 或预测值
# 再展开成 selected_rows、candidate_rows 等自定义表格。

for subgroup in SUBGROUPS:
    # df_atlas 与 FC_all 严格按行对齐：第 i 行对应同一名被试。
    df = df_atlas.copy()

    # 无法解析的数值会变成 NaN，并在共同筛选条件中被排除。
    for c in COVARIATE_COLS + [IAT_LABEL, CBCL_TOTAL_COL]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 所有 subgroup 共用的完整性和质量筛选。
    analysis_ready_mask = (
        df[COVARIATE_COLS + [IAT_LABEL, CBCL_TOTAL_COL, SITE_COL]].notna().all(axis=1).to_numpy()
        & np.isfinite(df[IAT_LABEL].to_numpy(dtype=float))
        & np.isfinite(df[MOTION_COL].to_numpy(dtype=float))
        & (df[MOTION_COL].to_numpy(dtype=float) < ENORM_CUTOFF)
        & (df[CBCL_TOTAL_COL].to_numpy(dtype=float) < CBCL_CUTOFF)
    )

    # 当前 IAT subgroup 的筛选条件。
    if subgroup == "highIAT":
        subgroup_mask = df[IAT_LABEL].to_numpy(dtype=float) > IAT_CUTOFF
    else:
        subgroup_mask = df[IAT_LABEL].to_numpy(dtype=float) <= IAT_CUTOFF

    subgroup_analysis_mask = analysis_ready_mask & subgroup_mask

    # 组织 cpm_nested_cv_parallel() 输入。
    y_sub = df.loc[subgroup_analysis_mask, IAT_LABEL].to_numpy(dtype=float)
    X_sub = FC_all[subgroup_analysis_mask].astype(np.float32, copy=False)
    cov_basic = df.loc[subgroup_analysis_mask, COVARIATE_COLS].to_numpy(dtype=float)
    site_dummies = pd.get_dummies(
        df.loc[subgroup_analysis_mask, SITE_COL].astype(str),
        prefix=SITE_COL,
        drop_first=True,
    )
    cov_sub = np.column_stack([cov_basic, site_dummies.to_numpy(dtype=float)])

    # 同步排除 FC 中含 NaN/Inf 的被试，保证 X、y 和协变量仍按行对应。
    finite_fc_mask = np.isfinite(X_sub).all(axis=1)
    X_sub = X_sub[finite_fc_mask]
    y_sub = y_sub[finite_fc_mask]
    cov_sub = cov_sub[finite_fc_mask]

    print(f"\n[{subgroup}] N={len(y_sub)}, n_edges={X_sub.shape[1]}, covariates={cov_sub.shape[1]}")
    print(f"IAT mean={np.mean(y_sub):.3f}, IAT sd={np.std(y_sub):.3f}")
    print("Site dummy columns:", list(site_dummies.columns))

    # cpm_nested_cv_parallel() 并行处理 outer folds；每个 fold 内部完成：
    # 1. 协变量残差化；
    # 2. CPM 选边；
    # 3. inner KFold 选择 ridge_alpha；
    # 4. 用最佳 alpha 重训并预测完全未参与调参的 outer test subject。
    cv_results = cpm_nested_cv_parallel(
        X=X_sub,
        y=y_sub,
        Covariates=cov_sub,
        param_grid={"ridge_alpha": ALPHA_GRID},
        cv_type="loocv",
        n_repeats=1,
        inner_cv_type="kfold",
        inner_n_splits=INNER_N_SPLITS,
        scoring="pearson_r",
        selection_method="pvalue",
        p_thresh=P_THRESH,
        model_type="ridge",
        standardize_features=STANDARDIZE_FEATURES,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=True,
    )

    # cv_results 完全保留 cpm_nested_cv() 的六元素 tuple：
    # [0] y_pred_mean
    # [1] y_resid_true_mean
    # [2] y_pred_mat
    # [3] y_resid_true_mat
    # [4] edge_records
    # [5] selected_param_records
    performance = calculate_prediction_performance(
        y_resid_true_mat=cv_results[3],
        y_pred_mat=cv_results[2],
    )

    # 分 subgroup 直接保存原始对象。pickle 不改变 tuple、dict、ndarray
    # 或 list-of-dict 的层次，因此重新读取后仍是 CPM 模块熟悉的输出结构。
    cv_path = RESULT_DIR / f"CPM_nestedCV_{subgroup}_cv_results.pkl"
    performance_path = RESULT_DIR / f"CPM_nestedCV_{subgroup}_performance.pkl"
    with open(cv_path, "wb") as f:
        pickle.dump(cv_results, f)
    with open(performance_path, "wb") as f:
        pickle.dump(performance, f)

    print("Performance:")
    print(performance)
    print("Saved:", cv_path)
    print("Saved:", performance_path)

    print(f"[{subgroup}] finished")



[highIAT] N=461, n_edges=22791, covariates=9
IAT mean=41.534, IAT sd=9.244
Site dummy columns: ['ScannerSite_CUNY', 'ScannerSite_RU']
使用 n_jobs=20 并行计算 461 个 nested CV 外层折。
Performance:
{'per_repeat': {'pearson_r': array([0.26517313]), 'r2': array([-0.65257583]), 'mae': array([8.61136632]), 'rmse': array([11.08992038])}, 'summary': {'n_repeats': 1, 'n_subjects': 461, 'pearson_r_mean': 0.2651731290463208, 'pearson_r_std': 0.0, 'r2_mean': -0.6525758281640619, 'r2_std': 0.0, 'mae_mean': 8.611366316906167, 'mae_std': 0.0, 'rmse_mean': 11.089920380753457, 'rmse_std': 0.0, 'pearson_r_fisher_mean': 0.2651731290463208}, 'mean_prediction': {'pearson_r': 0.2651731290463208, 'r2': -0.6525758281640619, 'mae': 8.611366316906167, 'rmse': 11.089920380753457}}
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV/CPM_nestedCV_highIAT_cv_results.pkl
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV/CPM_nestedCV_highIAT_performance.pkl
[highIAT] finished

[lowIAT] N=653, n_edges=22791, covariat

/Users/fmri24/fsl/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Performance:
{'per_repeat': {'pearson_r': array([0.28458225]), 'r2': array([0.0283602]), 'mae': array([6.67268701]), 'rmse': array([8.1433056])}, 'summary': {'n_repeats': 1, 'n_subjects': 653, 'pearson_r_mean': 0.28458225307848883, 'pearson_r_std': 0.0, 'r2_mean': 0.028360197795670894, 'r2_std': 0.0, 'mae_mean': 6.672687010161448, 'mae_std': 0.0, 'rmse_mean': 8.143305603075909, 'rmse_std': 0.0, 'pearson_r_fisher_mean': 0.28458225307848883}, 'mean_prediction': {'pearson_r': 0.28458225307848883, 'r2': 0.028360197795670894, 'mae': 6.672687010161448, 'rmse': 8.143305603075909}}
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV/CPM_nestedCV_lowIAT_cv_results.pkl
Saved: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV/CPM_nestedCV_lowIAT_performance.pkl
[lowIAT] finished


## 5. 保存 nested 结果

每个 subgroup 保存两个 pickle 文件：

- `*_cv_results.pkl`：nested CV 返回的原始六元素 tuple；
- `*_performance.pkl`：`calculate_prediction_performance()` 返回的原始 dict。

`pickle.load()` 读取后会恢复函数返回时的原始对象结构。


In [6]:
# 读取示例（按需取消注释）
# with open(RESULT_DIR / "CPM_nestedCV_highIAT_cv_results.pkl", "rb") as f:
#     cv_results = pickle.load(f)
# with open(RESULT_DIR / "CPM_nestedCV_highIAT_performance.pkl", "rb") as f:
#     performance = pickle.load(f)

print("Done. Nested CV outputs are in:", RESULT_DIR)


Done. Nested CV outputs are in: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_LOOCV
